In [1]:
import warnings
warnings.simplefilter(action='ignore')

import os
import requests
import joblib

import numpy
import pandas

import matplotlib.pyplot as plt
import scipy.stats as stats

In [2]:
from catboost import CatBoostRegressor ,Pool
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

from sklearn.linear_model import ElasticNetCV, LassoCV, RidgeCV
from sklearn.svm import SVR

from sklearn.ensemble import GradientBoostingRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import StackingRegressor
from sklearn.ensemble import AdaBoostRegressor
from sklearn.ensemble import ExtraTreesRegressor

from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import RobustScaler
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MinMaxScaler

from sklearn.metrics import mean_squared_error
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import accuracy_score

from sklearn.pipeline import make_pipeline
from sklearn.model_selection import KFold, cross_val_score

from sklearn.feature_selection import SelectKBest
from sklearn.feature_selection import f_classif
from sklearn.feature_selection import SelectFromModel

from sklearn.impute import SimpleImputer
import sklearn.linear_model as linear_model
from sklearn.model_selection import train_test_split
import seaborn as sns
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

In [3]:
train = pandas.read_csv('/kaggle/input/zelestra-x-aws-ml-ascend-challenge/train.csv')
test = pandas.read_csv('/kaggle/input/zelestra-x-aws-ml-ascend-challenge/test.csv').drop(columns=['id'])

In [4]:
encoder = LabelEncoder()
normalize = SimpleImputer(strategy='mean')

In [5]:
for i in zip(train.columns,train.dtypes):
    if i[1]=='O':
        train[i[0]] = train[i[0]].fillna('Unknown')
        train[i[0]]=normalize.fit_transform(encoder.fit_transform(train[i[0]].to_numpy().reshape(-1,1)).reshape(-1,1))
    else:
        train[i[0]].fillna(train[i[0]].mean(), inplace=True)
        train[i[0]]=normalize.fit_transform(numpy.array(train[i[0]]).reshape(-1,1))
for i in zip(test.columns,test.dtypes):
    if i[1]=='O':
        test[i[0]] = test[i[0]].fillna('Unknown')
        test[i[0]]=normalize.fit_transform(numpy.array(encoder.fit_transform(test[i[0]].to_numpy().reshape(-1,1))).reshape(-1,1))
    else:
        test[i[0]].fillna(test[i[0]].mean(), inplace=True)
        test[i[0]]=normalize.fit_transform(numpy.array(test[i[0]]).reshape(-1,1))

In [6]:
train_x = train.drop(columns=['id', 'efficiency'])
train_y = train['efficiency']

In [7]:
LGBM_R_parm = {'colsample_bytree': 0.7127886176981258,
               'drop_rate': 0.8124406739963222,
               'learning_rate': 0.07912182737635276,
               'max_bin': 6651,
               'max_depth': 9050,
               'max_drop': 8493,
               'min_child_samples': 6754,
               'min_data_in_leaf': 2929,
               'n_estimators': 25802,
               'num_leaves': 61,
               'objective': 'regression_l1',
               'reg_alpha': 0.881842406619579,
               'reg_lambda': 0.456201832946564,
               'skip_drop': 0.07580175182420101,
               'verbosity': -1}

XGB_R_parm = {'colsample_bytree': 0.6146156360042275,
              'gamma': 1,
              'learning_rate': 0.7129688111132676,
              'max_depth': 5,
              'min_child_weight': 1,
              'n_estimators': 3145,
              'objective': 'binary:logistic',
              'reg_alpha': 85,
              'reg_lambda': 0.27992509484698325,
              'subsample': 0.7284972930896713}

catboost_params = {'iterations' : 1000,
                   'learning_rate': 0.009, 
                   'depth': 7, 
                   'l2_leaf_reg': 4, 
                   'od_wait' : 50,
                   'random_state' : 42,
                   'eval_metric': 'RMSE', 
                   'iterations': 1000,
                   'bootstrap_type': 'Bayesian', 
                   'allow_const_label': True, 
                   'grow_policy' : 'Depthwise',
                   'logging_level' : 'Silent'
                 }

X_train, X_test, y_train, y_test = train_test_split(train_x, train_y, test_size=0.1, random_state=100)

In [8]:
LGBM_R = LGBMRegressor(**LGBM_R_parm)

In [9]:
XGB_R = XGBRegressor(**XGB_R_parm)

In [10]:
catboost = CatBoostRegressor(**catboost_params)

In [11]:
estimators = [
    ('LGBM', LGBM_R),
    ('xgb', XGB_R),
    ('cat', catboost)
]

model = StackingRegressor(estimators, final_estimator = RidgeCV())
model.fit(X_train, y_train)

StackingRegressor(estimators=[('LGBM',
                               LGBMRegressor(colsample_bytree=0.7127886176981258,
                                             drop_rate=0.8124406739963222,
                                             learning_rate=0.07912182737635276,
                                             max_bin=6651, max_depth=9050,
                                             max_drop=8493,
                                             min_child_samples=6754,
                                             min_data_in_leaf=2929,
                                             n_estimators=25802, num_leaves=61,
                                             objective='regression_l1',
                                             reg_alpha=0.881842406619579,
                                             reg_lambda=0.456201832946...
                                            max_bin=None,
                                            max_cat_threshold=None,
                                            max_cat_to_onehot=None,
                                            max_delta_step=None, max_depth=5,
                                            max_leaves=None, min_child_weight=1,
                                            missing=nan,
                                            monotone_constraints=None,
                                            multi_strategy=None,
                                            n_estimators=3145, n_jobs=None,
                                            num_parallel_tree=None,
                                            objective='binary:logistic', ...)),
                              ('cat',
                               <catboost.core.CatBoostRegressor object at 0x7f1499342590>)],
                  final_estimator=RidgeCV())

In [12]:
print(f'Concordance index (lifelines): {100*(1-numpy.sqrt(mean_squared_error(y_test, model.predict(X_test))))}')

Concordance index (lifelines): 90.24897778192343


In [13]:
id = pandas.read_csv('/kaggle/input/zelestra-x-aws-ml-ascend-challenge/test.csv')['id']
test_predictions = model.predict(test)
test_predictions

array([0.39852307, 0.5451521 , 0.51996381, ..., 0.62817857, 0.42968259,
       0.56334105])

In [14]:
submission = pandas.DataFrame({
    'id': id.values,
    'efficiency': test_predictions,
})
submission

,id,efficiency
0,0,0.398523
1,1,0.545152
2,2,0.519964
3,3,0.470968
4,4,0.451675
...,...,...
11995,11995,0.555283
11996,11996,0.462189
11997,11997,0.628179
11998,11998,0.429683


In [15]:
submission.to_csv('submission.csv', index = False)